# 🔧 Support Vector Machines
**ISLP Chapter 9 · Data Pattern Recognition for the Rest of Us**

> SVMs find the hyperplane that maximally separates classes. With kernels, they handle nonlinear boundaries. They work well in high dimensions and with imbalanced classes — making them a natural fit for fraud detection.

### Dataset
**Credit Card Fraud Detection**
Synthetic dataset modelled on the properties of the real European credit card fraud dataset (Kaggle). 284,807 transactions, 0.17% fraud rate — a classic imbalanced classification challenge in financial services.

### Time: ~60 minutes

In [ ]:
# ⚠️  Run this cell first — sets up data and imports
# Tip: Runtime → Run all  (runs everything top to bottom automatically)

import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fa',
    'axes.grid': True, 'grid.alpha': 0.4,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11
})

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, RocCurveDisplay
from sklearn.decomposition import PCA

# ── Load / generate data ──────────────────────────────────────────────────────
# Credit card fraud detection (inline)
np.random.seed(42)
n = 5000
legit  = np.random.multivariate_normal([0,0], [[1,.3],[.3,1]], int(n*0.98))
fraud  = np.random.multivariate_normal([2,2], [[0.5,.1],[.1,0.5]], int(n*0.02))
X_svm  = np.vstack([legit, fraud])
y_svm  = np.array([0]*len(legit) + [1]*len(fraud))
svm_df = pd.DataFrame(X_svm, columns=["feature1","feature2"])
svm_df["fraud"] = y_svm
print(f"✓ Fraud dataset: {svm_df.shape}  Fraud rate: {svm_df.fraud.mean():.1%}")

print(f'  Python {sys.version.split()[0]} | pandas {pd.__version__}')
print('✓ Setup complete')


In [ ]:
# Credit Card Fraud — realistic synthetic dataset
# Properties match the real Kaggle dataset:
# - Highly imbalanced (0.17% fraud)
# - PCA-transformed features (V1-V8) + Amount + Time
np.random.seed(42)
n_legit, n_fraud = 10000, 17  # ~0.17% fraud rate

# Legitimate transactions
legit = np.random.multivariate_normal(
    mean=np.zeros(8),
    cov=np.eye(8) + 0.2*np.random.rand(8,8),
    size=n_legit)
legit_amount = np.random.lognormal(4.5, 1.2, n_legit)

# Fraudulent transactions — different distribution
fraud = np.random.multivariate_normal(
    mean=[2,-2,1.5,-1.5,0.8,-0.8,1,-0.5],
    cov=np.eye(8)*0.5,
    size=n_fraud)
fraud_amount = np.random.lognormal(3.8, 0.8, n_fraud)

X_raw = np.vstack([legit, fraud])
amounts = np.concatenate([legit_amount, fraud_amount])
y_fraud = np.array([0]*n_legit + [1]*n_fraud)

# Scale amount (like the real dataset)
scaler_amt = StandardScaler()
X_full = np.column_stack([X_raw, scaler_amt.fit_transform(amounts.reshape(-1,1))])

print(f"Credit card transactions: {len(y_fraud):,}")
print(f"Legitimate: {n_legit:,}  ({n_legit/len(y_fraud)*100:.1f}%)")
print(f"Fraudulent: {n_fraud:,}  ({n_fraud/len(y_fraud)*100:.2f}%)")
print(f"\nClass imbalance ratio: {n_legit/n_fraud:.0f}:1")
print("\nThis is why accuracy is a TERRIBLE metric here:")
print(f"  Always predicting 'legit' gives {n_legit/len(y_fraud)*100:.2f}% accuracy")
print("  but catches exactly 0% of fraud!")

## 🎯 Part 1 — The Maximal Margin Classifier

SVM finds the hyperplane that **maximizes the margin** — the distance to the nearest training points (support vectors) from each class.

```
Maximize M  subject to:  yᵢ(β₀ + β₁xᵢ₁ + ... + βₚxᵢₚ) ≥ M
```

The **soft margin** (C parameter) allows some misclassification:
- Large C → narrow margin, few violations, may overfit
- Small C → wide margin, tolerates more violations, more robust

For fraud detection: we want high **recall on fraud** (catch as many as possible) even at the cost of some false alarms.

In [ ]:
# Use 2 most discriminating features for visualization
X_tr, X_te, y_tr, y_te = train_test_split(X_full, y_fraud, test_size=0.25,
                                             random_state=42, stratify=y_fraud)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s  = scaler.transform(X_te)

# Visualize decision boundary using top 2 features
X_2d = X_tr_s[:, :2]
y_2d = y_tr

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
C_vals   = [0.1, 1.0, 100.0]
titles   = ['Small C=0.1\n(wide margin, tolerant)',
            'C=1.0\n(balanced)',
            'Large C=100\n(tight margin, strict)']

for ax, C, title in zip(axes, C_vals, titles):
    svm = SVC(kernel='rbf', C=C, probability=True, class_weight='balanced')
    svm.fit(X_2d, y_2d)

    xx, yy = np.meshgrid(np.linspace(-4,4,200), np.linspace(-4,4,200))
    Z = svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.2, colors=['#1e5fa8','#e85d20'])
    ax.scatter(X_2d[y_2d==0,0], X_2d[y_2d==0,1], color='#1e5fa8',
               alpha=0.3, s=8, label='Legit')
    ax.scatter(X_2d[y_2d==1,0], X_2d[y_2d==1,1], color='#e85d20',
               s=80, marker='x', lw=2, label='Fraud')

    fraud_recall = svm.score(X_2d[y_2d==1], y_2d[y_2d==1])
    ax.set_title(f'{title}\nFraud recall={fraud_recall:.0%}', fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle('SVM Decision Boundaries — Credit Card Fraud\n'
             'Orange X marks = actual fraud transactions',
             fontsize=12, y=1.02)
plt.tight_layout(); plt.show()

## 🔮 Part 2 — The Kernel Trick

For nonlinear fraud patterns, the **RBF (Radial Basis Function) kernel** maps data to a higher-dimensional space where a linear boundary separates classes.

```
K(x,z) = exp(-γ||x-z||²)
```
γ controls how far a single training example's influence reaches — higher γ = tighter fit.

In [ ]:
# Full model with all features — RBF kernel, class_weight to handle imbalance
svm_full = SVC(kernel='rbf', C=1.0, gamma='scale',
               class_weight='balanced', probability=True, random_state=42)
svm_full.fit(X_tr_s, y_tr)
y_pred = svm_full.predict(X_te_s)
y_prob = svm_full.predict_proba(X_te_s)[:,1]

print("=== SVM on Credit Card Fraud ===\n")
print(classification_report(y_te, y_pred, target_names=['Legit','Fraud'],
                            digits=4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cm = confusion_matrix(y_te, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Legit','Fraud']).plot(
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix\n(we care most about fraud recall)')

RocCurveDisplay.from_estimator(svm_full, X_te_s, y_te, ax=axes[1])
axes[1].set_title(f'ROC Curve (AUC={roc_auc_score(y_te,y_prob):.4f})')
plt.tight_layout(); plt.show()

n_sv = svm_full.n_support_.sum()
print(f"\n\u2714 Number of support vectors: {n_sv}")
print(f"   These are the only transactions that define the boundary")
print(f"   All other transactions can be removed without changing the model")

In [ ]:
# Business framing: threshold tuning
# In fraud detection, missing fraud (false negative) is far more costly than
# a false alarm (false positive). Adjust threshold to maximize fraud recall.
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_te, y_prob)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thresholds, precision[:-1], color='#1e5fa8', lw=2, label='Precision')
ax.plot(thresholds, recall[:-1],    color='#e85d20', lw=2, label='Recall (fraud caught)')
ax.axvline(0.5, color='#888', lw=1.5, ls='--', label='Default threshold = 0.5')
ax.set_xlabel('Classification Threshold')
ax.set_ylabel('Score')
ax.set_title('Precision-Recall Tradeoff\n'
             'Lower threshold = catch more fraud but more false alarms')
ax.legend()
plt.tight_layout(); plt.show()

print("\n\u2714 Business decision: fraud cost >> false alarm cost")
print("   A threshold of 0.2-0.3 may be more appropriate than 0.5")
print("   This is a BUSINESS decision, not a modeling one")

In [ ]:
# @title 📝 Quiz — Support Vector Machines
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What are support vectors and why do only they define the boundary?
# @markdown **Q2:** What does the C parameter control and what happens at extremes?
# @markdown **Q3:** Why is accuracy a bad metric for fraud detection?
# @markdown **Q4:** What is the kernel trick and why does RBF work well for fraud?
# @markdown **Q5:** Should you lower the fraud threshold from 0.5? Justify your answer.

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

### 📤 Submit for AI Grading

In [ ]:
_NB_NAME_="Support Vector Machines"
# @title 📋 Quiz Submission
# @markdown **Click ▶ Run** → copy the output box (click the copy icon in the top-left of the output box, or click ⋮ → Copy) → paste into Gemini in this Colab session.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("Run the quiz cell above first, then re-run this cell.")
else:
    qa = "\n".join(
        f"Q{i}: {str(v).strip() or '(no answer)'}"
        for i, (_, v) in enumerate(answers.items(), 1)
    )
    print(f"Please grade my quiz answers for the \"{_NB_TITLE}\" notebook")
    print(f"from \"Data Pattern Recognition for the Rest of Us\" (based on ISLP).")
    if GITHUB_USERNAME.strip():
        print(f"Student: @{GITHUB_USERNAME.strip()}")
    print()
    print(qa)
    print()
    print("For each question:")
    print("  1. Say CORRECT, PARTIAL, or INCORRECT")
    print("  2. Explain in 2-3 sentences why — what did I get right or wrong")
    print("  3. Give the ideal complete answer so I know exactly what was expected")
    print("  4. If I was wrong or partial, tell me the specific concept to review")
    print("     and where it appears in the notebook (e.g. Part 2, the SHAP beeswarm plot)")
    print()
    print("End with:")
    print("  - Overall grade: Excellent / Good / Needs Review / Incomplete")
    print("  - A short study plan: which questions to re-read and what to focus on")
    print()
    print("After grading, I will paste specific outputs, charts, or code blocks")
    print("from the notebook — please explain them in detail and answer follow-up questions.")


---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md) — describe the notebook name, cell number, and what went wrong. This is how real open-source projects work.
*Search [existing issues](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues) first to avoid duplicates.*

---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*